# 01 · Ingest: raw Datashop exports → one event log

**Input:** 13 compressed Runestone Datashop exports in `data/raw/<semester>/` (not in git; see `data/raw/README.md`).
**Output:** `data/interim/events.parquet`, one row per logged ebook event, plus row-count and data-quality tables.

What this stage does, in order:
1. Verify each raw file against `data/raw/MANIFEST.csv` (sha256), so results are tied to an exact set of inputs.
2. Read each export as text, normalize the column names (the exports drift between semesters).
3. **Namespace student and session ids by semester and section.** Each section's export numbers students from 1, so a bare id is not unique.
4. Write one sorted parquet with DuckDB, then run SQL checks on it.

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

## 1. The manifest: which files, which semester, which checksum

In [2]:
from so_ell import ingest
show(ingest.infer_section, ingest.build_manifest)
manifest = pd.read_csv(config.RAW_MANIFEST)
manifest[["semester", "section", "path", "bytes"]]

`so_ell.ingest.infer_section`
```python
def infer_section(file_name: str) -> str:
    name = file_name.lower()
    if re.search(r"(^|[-_])mw([-_]|$)", name):
        return "MW"
    if "tth" in name or "t-thurs" in name:
        return "TTh"
    return "ALL"
```

`so_ell.ingest.build_manifest`
```python
def build_manifest(raw_dir: Path = config.RAW_DIR) -> pd.DataFrame:
    """Scan data/raw/<semester>/ and describe every compressed export.

    Run once (``so-ell manifest``) when raw files change, then commit the CSV.
    """
    rows = []
    for path in sorted(raw_dir.glob("*/*")):
        if path.suffix not in {".gz", ".zip", ".bz2"}:
            continue
        semester = path.parent.name
        rows.append(
            {
                "semester": semester,
                "section": infer_section(path.name),
                "path": path.relative_to(raw_dir).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
        )
    manifest = pd.DataFrame(rows)
    manifest.to_csv(raw_dir / "MANIFEST.csv", index=False)
    return manifest
```

,semester,section,path,bytes
0,F21,ALL,F21/F21-py4e-int_datashop.txt.gz,19017830
1,F22,ALL,F22/Fall22-datashop.csv.gz,42244534
2,F23,MW,F23/MW-py4e-int_datashop-4.tab.gz,10996713
3,F23,TTh,F23/T-Thurs-py4e-int_datashop-4.tab.gz,26360461
4,F24,MW,F24/F24-SI206-MW_datashop_2025-05-30-13-39.tab...,16544660
5,F24,TTh,F24/F24-SI206-TTh_datashop_2025-05-22-14-31.ta...,28013107
6,F25,MW,F25/F25-SI201-MW_datashop_2026-02-03-22-47.tab...,39395339
7,F25,TTh,F25/F25-SI201-TTh_datashop_2026-02-03-23-02.ta...,51232670
8,W20,ALL,W20/SI206-win20-py4e-int_datashop.tab.zip,6630
9,W21,ALL,W21/Win21-SI206_datashop_2026-02-04-16-05.tab.zip,503


## 2. Reading one export
Everything is read as text first (`dtype=str`) so nothing is silently coerced; ids are rebuilt as `F25-MW-0005`.

In [3]:
show(ingest.standardize_columns, ingest.read_export)

`so_ell.ingest.standardize_columns`
```python
def standardize_columns(columns: pd.Index) -> list[str]:
    return (
        columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .tolist()
    )
```

`so_ell.ingest.read_export`
```python
def read_export(path: Path, semester: str, section: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=True, compression="infer")
    df.columns = standardize_columns(df.columns)
    missing = [c for c in CANONICAL_COLUMNS if c not in df.columns]
    for col in missing:
        df[col] = pd.NA
    df = df[CANONICAL_COLUMNS].copy()

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    for col in INT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    local_student = pd.to_numeric(df["anon_student_id"], errors="coerce").astype("Int64")
    local_session = pd.to_numeric(df["session_id"], errors="coerce").astype("Int64")
    prefix = f"{semester}-{section}-"
    df["local_student_id"] = local_student
    df["anon_student_id"] = (prefix + local_student.astype("string").str.zfill(4)).astype("string")
    df["session_id"] = (prefix + local_session.astype("string")).astype("string")
    df["semester"] = semester
    df["section"] = section
    df["source_file"] = path.name
    return df
```

## 3. Run the ingest
Checksums are verified, each file is written as a parquet part, and DuckDB concatenates and sorts them by semester, student and time.

In [4]:
show(ingest.run_ingest)
counts = ingest.run_ingest(verify_checksums=True)
counts.to_csv(config.QUALITY_DIR / "ingest_row_counts.csv", index=False)
counts

`so_ell.ingest.run_ingest`
```python
def run_ingest(
    raw_dir: Path = config.RAW_DIR,
    out_path: Path = config.EVENTS_PARQUET,
    verify_checksums: bool = True,
) -> pd.DataFrame:
    """Read every manifest entry and write the combined event log.

    Returns a per-file row-count table (also written to outputs/data_quality).
    """
    manifest = pd.read_csv(raw_dir / "MANIFEST.csv")
    parts_dir = out_path.parent / "_event_parts"
    parts_dir.mkdir(parents=True, exist_ok=True)
    counts = []
    for row in manifest.itertuples(index=False):
        path = raw_dir / row.path
        if not path.exists():
            raise FileNotFoundError(f"{path} is listed in MANIFEST.csv but missing. See data/README.md.")
        if verify_checksums and sha256(path) != row.sha256:
            raise ValueError(f"Checksum mismatch for {path}; the raw export changed.")
        print(f"  reading {row.path}", flush=True)
        df = read_export(path, row.semester, row.section)
        part = parts_dir / f"{row.semester}_{row.section}.parquet"
        df.to_parquet(part, index=False)
        counts.append(
            {
                "semester": row.semester,
                "section": row.section,
                "source_file": path.name,
                "rows": len(df),
                "students": df["anon_student_id"].nunique(),
                "null_time_rows": int(df["time"].isna().sum()),
            }
        )
        del df

    # DuckDB does the concat + sort out of core, so this works on a laptop.
    con = duckdb.connect()
    con.execute("SET enable_progress_bar = false")
    parts_glob = (parts_dir / "*.parquet").as_posix()
    target = out_path.as_posix()
    con.execute(
        f"""
        COPY (
            SELECT * FROM read_parquet('{parts_glob}', union_by_name = true)
            ORDER BY semester, anon_student_id, time
        ) TO '{target}' (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 500000)
        """
    )
    con.close()
    for part in parts_dir.glob("*.parquet"):
        part.unlink()
    parts_dir.rmdir()
    return pd.DataFrame(counts)
```

  reading F21/F21-py4e-int_datashop.txt.gz


  reading F22/Fall22-datashop.csv.gz


  reading F23/MW-py4e-int_datashop-4.tab.gz


  reading F23/T-Thurs-py4e-int_datashop-4.tab.gz


...

  reading W21/Win21-SI206_datashop_2026-02-04-16-05.tab.zip


  reading W22/win22_si206_datashop.txt.bz2


  reading W23/Win23-SI206-datashop.csv.zip


  reading W24/Win24-SI206_datashop_2026-02-04-15-54.tab.zip


,semester,section,source_file,rows,students,null_time_rows
0,F21,ALL,F21-py4e-int_datashop.txt.gz,1285969,114,4719
1,F22,ALL,Fall22-datashop.csv.gz,2195138,159,0
2,F23,MW,MW-py4e-int_datashop-4.tab.gz,559272,76,0
3,F23,TTh,T-Thurs-py4e-int_datashop-4.tab.gz,1328973,154,0
4,F24,MW,F24-SI206-MW_datashop_2025-05-30-13-39.tab.zip,844291,89,0
5,F24,TTh,F24-SI206-TTh_datashop_2025-05-22-14-31.tab.zip,1383038,156,0
6,F25,MW,F25-SI201-MW_datashop_2026-02-03-22-47.tab.zip,3224307,112,0
7,F25,TTh,F25-SI201-TTh_datashop_2026-02-03-23-02.tab.zip,4221778,129,0
8,W20,ALL,SI206-win20-py4e-int_datashop.tab.zip,404,14,0
9,W21,ALL,Win21-SI206_datashop_2026-02-04-16-05.tab.zip,5,1,0


In [5]:
print(f"{counts['rows'].sum():,} events in the combined log")

19,805,383 events in the combined log


W20 and W21 are near-empty exports (404 and 5 rows, no timed exams), which is why the analysis covers 8 semesters:

In [6]:
print("analysis semesters:", config.ANALYSIS_SEMESTERS, "| excluded:", config.EXCLUDED_SEMESTERS)

analysis semesters: ['F21', 'W22', 'F22', 'W23', 'F23', 'W24', 'F24', 'F25'] | excluded: {'W20': 'export has 404 rows and no timed exams', 'W21': 'export has 5 rows'}


## 4. Data-quality checks on the event log
*Hard* checks fail the pipeline; *soft* checks are reported.

In [7]:
from so_ell import validation
show(validation.event_log_checks)
checks = validation.event_log_checks()
checks.to_csv(config.QUALITY_DIR / "event_log_checks.csv", index=False)
checks

`so_ell.validation.event_log_checks`
```python
def event_log_checks(events_path: Path = config.EVENTS_PARQUET) -> pd.DataFrame:
    """Row-level checks on the full event log, run in DuckDB. Returns a pass/fail table."""
    con = duckdb.connect()
    con.execute("SET enable_progress_bar = false")
    src = f"read_parquet('{events_path.as_posix()}')"
    checks = {
        "no_null_student_id": f"SELECT COUNT(*) FROM {src} WHERE anon_student_id IS NULL",
        "student_id_format": f"SELECT COUNT(*) FROM {src} WHERE NOT regexp_matches(anon_student_id, '{STUDENT_ID}')",
        "no_null_selection": f"SELECT COUNT(*) FROM {src} WHERE selection IS NULL",
        "time_parsed": f"SELECT COUNT(*) FROM {src} WHERE time IS NULL",
        "time_in_plausible_range": f"SELECT COUNT(*) FROM {src} WHERE time < TIMESTAMP '2019-01-01' OR time > TIMESTAMP '2026-12-31'",
        "exact_duplicate_rows": f"""SELECT COUNT(*) - COUNT(DISTINCT (time, anon_student_id, action, problem_name, session_id, selection, input))
                                    FROM {src}""",
        "event_outside_semester_window": f"""
            WITH s AS (
                SELECT semester, quantile_cont(time, 0.001) lo, quantile_cont(time, 0.999) hi
                FROM {src} GROUP BY semester
            )
            SELECT COUNT(*) FROM {src} e JOIN s USING (semester)
            WHERE e.time < s.lo - INTERVAL 60 DAY OR e.time > s.hi + INTERVAL 60 DAY""",
    }
    # Some rows legitimately fail soft checks (unparseable timestamps exist in the
    # raw export). Hard checks must be zero; soft checks are reported, not fatal.
    hard = {"no_null_student_id", "student_id_format", "no_null_selection"}
    total = con.execute(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    rows = []
    for name, sql in checks.items():
        failures = con.execute(sql).fetchone()[0]
        rows.append(
            {
                "check": name,
                "severity": "hard" if name in hard else "soft",
                "failing_rows": int(failures),
                "failing_share": failures / total,
                "passed": failures == 0 or name not in hard,
            }
        )
    con.close()
    report = pd.DataFrame(rows)
    failed = report.loc[~report["passed"], "check"].tolist()
    if failed:
        raise ValueError(f"Event log failed hard checks: {failed}")
    return report
```

,check,severity,failing_rows,failing_share,passed
0,no_null_student_id,hard,0,0.000000,True
1,student_id_format,hard,0,0.000000,True
2,no_null_selection,hard,0,0.000000,True
3,time_parsed,soft,4719,0.000238,True
4,time_in_plausible_range,soft,0,0.000000,True
5,exact_duplicate_rows,soft,162263,0.008193,True
6,event_outside_semester_window,soft,1877,0.000095,True


## 5. Why the ids had to be namespaced

How many local ids appear in **both** section exports of a semester and started the real midterm in both?
In the original project these were merged into one "student".

In [8]:
import duckdb
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
EV = f"read_parquet('{config.EVENTS_PARQUET.as_posix()}')"
con.execute(f'''
WITH starts AS (
    SELECT semester, section, local_student_id
    FROM {EV}
    WHERE selection = 'timedExam' AND action = 'start'
      AND regexp_matches(lower(problem_name), 'mid')
      AND NOT regexp_matches(lower(problem_name), 'practice')
    GROUP BY ALL
)
SELECT semester, COUNT(*) AS local_ids_in_both_sections
FROM (SELECT semester, local_student_id FROM starts GROUP BY ALL HAVING COUNT(DISTINCT section) = 2)
GROUP BY semester ORDER BY semester
''').df()

,semester,local_ids_in_both_sections
0,F23,59
1,F25,87
